In [ ]:
import pandas as pd 
import nltk
nltk.download('universal_tagset')
from nltk.tag import pos_tag
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from googletrans import Translator
import time
import re
import gensim.downloader as api
import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


### Ask for language of the input data

The data will be translated into english and ran through pipeline

I am choosing this approach, because from the research I've done, there is not a single one module which supports multilingual pre-processing for all the required extractions. This would lead to combining many different packages and not even coming close to covering the most common languages. That is why I am choosing this simpler solution 

In [ ]:
selected_language = 'english'  # Specify the language of the input data here

In [ ]:
sentences = pd.read_csv(r'C:\Users\filip\Documents\GitHub\fae2-nlpr-group-group-12-1\Data\CSV\recording_sentences_AssemlyAI_10_9.csv')

In [ ]:
translator = Translator()

def translate_to_english(text, source_language):
    # Skip translation if source language is already English
    if source_language.lower() in ['en', 'english']:
        return text
    try:
        result = translator.translate(text, src=source_language, dest='en')
        return result.text
    except Exception as e:
        print(f"Translation error: {e}")
        time.sleep(1)  # Add delay if rate limited
        return text  # Return original if translation fails

# Apply to DataFrame
sentences['translated_text'] = sentences.apply(
    lambda row: translate_to_english(row['sentence_column'], selected_language), 
    axis=1
)

## Sentence related features

Part-of-Speech (POS) Tagging:
Extract POS tags for each sentence in your transcription using an NLP library such as SpaCy or NLTK.
Create a new column in your dataset named ‘POS_Tags’ and store the POS tags for each sentence.

Sentiment Analysis:
Perform sentiment analysis on each sentence using a library like TextBlob or VADER or another one that you find suiting for your data.
Add a column ‘Sentiment_Score’ to your dataset with the sentiment polarity score for each sentence.

Pretrained Word Embeddings:
Use pretrained word embeddings (e.g., Word2Vec, GloVe) to convert the sentences in your dataset into vector representations.
Describe what happens with words that are not present in the pre-trained word embedding model.

### POS tags

In [ ]:
def POS_tagging(sentences):
    sentences['POS_tags'] = sentences['Sentence'].apply(lambda x: pos_tag(word_tokenize(x), tagset='universal'))

### Sentiment

In [ ]:
def sentiment_analysis(sentences):
    sentences['Sentiment'] = sentences['Sentence'].apply(lambda x: TextBlob(x).sentiment.polarity)

### TF-IDF

In [ ]:
def tfidf_vectorization(sentences):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences['Sentence'])

    # Convert sparse matrix to dense and then to list of arrays
    tfidf_dense = tfidf_matrix.toarray()

    # Add the TF-IDF vectors as a new column
    sentences['TF-IDF'] = [row for row in tfidf_dense]


In [42]:
sentences

,Sentence,POS_tags,Sentiment,TF-IDF,word2vec_embedding,custom_word2vec_embedding,Mean TF-IDF
0,"Laverne, California.","[(Laverne, NOUN), (,, .), (California, NOUN), ...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107
1,"Just 30 miles from Los Angeles, this suburb is...","[(Just, ADV), (30, NUM), (miles, NOUN), (from,...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2950870124125...","[0.011533101, 0.05309367, 0.13485718, 0.086883...","[-0.057924256, 0.053009097, 0.02974523, 0.1359...",0.250000
2,"And in 2010, it was bought by the Leyva family.","[(And, CONJ), (in, ADP), (2010, NUM), (,, .), ...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.43236119930387173, 0.0,...","[0.052022297, 0.017326036, 0.07470703, 0.05369...","[-0.0005464767, 0.022454534, -0.020723721, 0.0...",0.316228
3,Would you like a booth?,"[(Would, VERB), (you, PRON), (like, ADP), (a, ...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.13232422, 0.09358724, 0.095199585, 0.214843...","[-0.06910963, -0.02379939, 0.073943675, 0.0718...",0.500000
4,I started working here 10 years ago.,"[(I, PRON), (started, VERB), (working, VERB), ...",0.0,"[0.42853089523033905, 0.0, 0.0, 0.0, 0.0, 0.0,...","[-0.0703125, 0.10585938, 0.014611816, 0.108007...","[-0.16886294, 0.09133559, -0.016162036, 0.0892...",0.408248
...,...,...,...,...,...,...,...
1033,But so is her relationship with her family.,"[(But, CONJ), (so, ADV), (is, VERB), (her, PRO...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.028438022, -0.012276785, 0.000919887, 0.027...","[0.05844805, -0.0071843183, -0.05650278, 0.122...",0.361403
1034,I owe Chef Ramsay like everything.,"[(I, PRON), (owe, VERB), (Chef, NOUN), (Ramsay...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.057617188, -0.029846191, 0.112781525, 0.10...","[-0.019047227, -0.003497335, -0.21012314, -0.1...",0.447214
1035,He has molded me into the business owner I nee...,"[(He, PRON), (has, VERB), (molded, VERB), (me,...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.007220459, 0.005822754, -0.009115601, -0.00...","[-0.093392804, 0.05914693, -0.011449378, 0.040...",0.301511
1036,"When he comes back, he's gonna be like, you've...","[(When, ADV), (he, PRON), (comes, VERB), (back...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.108947754, 0.008560181, 0.058883667, 0.1559...","[-0.16133578, -0.021032054, -0.03779584, 0.041...",0.279715


### Pretrained Word Embeddings

In [ ]:
# Load pretrained Word2Vec model
model = api.load("word2vec-google-news-300")

def word2vec_embedding(model, sentences):
    vectors = []
    all_missing_words = []
    
    for sentence in sentences['Sentence']:
        words = sentence.lower().split()
        word_vectors = []
        missing_words = []
        
        for word in words:
            try:
                word_vectors.append(model[word])
            except KeyError:
                missing_words.append(word)
        
        if word_vectors:
            # Average all word vectors in the sentence
            sentence_vector = np.mean(word_vectors, axis=0)
        else:
            # If no words found, return zero vector
            sentence_vector = np.zeros(300)
        
        vectors.append(sentence_vector)
        all_missing_words.extend(missing_words)

    return vectors, all_missing_words

vectors, all_missing_words = word2vec_embedding(model, sentences)
# Add the vectors as a new column
sentences['word2vec_embedding'] = vectors

### Custom Word Embeddings

In [ ]:
reviews = pd.read_csv(r"C:\Users\filip\Documents\GitHub\fae2-nlpr-group-group-12-1\Data\CSV\Yelp Restaurant Reviews.csv")ň

In [ ]:
def yelp_custom_embeddings(reviews):
    reviews = reviews.drop(columns=['Yelp URL','Rating','Date'])
    reviews['Review Text'] = reviews['Review Text'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', '', x))
    reviews['Review Text'] = reviews['Review Text'].apply(lambda x: re.sub(r'\S+@\S+', '', x))
    reviews['Review Text'] = reviews['Review Text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))
    raw_corpus = reviews['Review Text'].tolist()
    return raw_corpus

In [ ]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs, email addresses, special characters
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

def tokenize_corpus(corpus):
    sentences = []
    
    for document in corpus:
        # Clean the document
        clean_doc = preprocess_text(document)
        
        # Split into sentences
        doc_sentences = sent_tokenize(clean_doc)
        
        for sentence in doc_sentences:
            # Tokenize words and remove very short sentences
            words = word_tokenize(sentence)
            if len(words) >= 3:  # Keep sentences with at least 3 words
                sentences.append(words)
    
    return sentences

# Preprocess your corpus
processed_sentences = tokenize_corpus(raw_corpus)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Corpus Statistics:
Total sentences: 19,892
Total words: 1,798,747
Vocabulary size: 27,186
Average sentence length: 90.4

Sample processed sentences:
Sentence 1: all i can say is they have very good ice cream i would for sure recommend their cookies and creme ice cream it is very good
Sentence 2: nice little local place for ice creammy favorite is their pumpkin shake fall season special my sweetness tolerance is low their large size ice cream usually seems too sweet after having ice cream for a while but love their pina colada so refreshing their banana split is good too
Sentence 3: a delicious treat on a hot day staff was very friendly and helpful gave us a sample and let us order a little earlier than open


In [ ]:
# Analyze corpus characteristics
total_sentences = len(processed_sentences)
total_words = sum(len(sentence) for sentence in processed_sentences)
vocab_size = len(set(word for sentence in processed_sentences for word in sentence))

print(f"Corpus Statistics:")
print(f"Total sentences: {total_sentences:,}")
print(f"Total words: {total_words:,}")
print(f"Vocabulary size: {vocab_size:,}")
print(f"Average sentence length: {total_words/total_sentences:.1f}")

# Show sample sentences
print(f"\nSample processed sentences:")
for i in range(3):
    print(f"Sentence {i+1}: {' '.join(processed_sentences[i])}")

In [ ]:
from gensim.models import Word2Vec
import multiprocessing

def train_word2vec_model(processed_sentences):
    
    # Set hyperparameters based on corpus analysis
    
    vector_size = 300      # 300 dimensions - good balance between expressiveness and efficiency
    window = 5            # 5 words context window - captures local semantic relationships
    min_count = 5         # Ignore words appearing less than 5 times - removes noise
    workers = multiprocessing.cpu_count()  # Use all CPU cores
    sg = 1                # Skip-gram model (sg=1) vs CBOW (sg=0)
    epochs = 30           # Number of training iterations
    alpha = 0.025         # Initial learning rate
    negative = 20         # Negative sampling - speeds up training
    
    # Train the Word2Vec model

    model = Word2Vec(
        sentences=processed_sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg,
        epochs=epochs,
        alpha=alpha,
        negative=negative
        )
    return model, vector_size

In [ ]:


custom_vectors = []
custom_missing_words = []

for sentence in sentences['Sentence']:
    words = sentence.lower().split()
    word_vectors = []
    missing_words = []
    
    for word in words:
        try:
            word_vectors.append(model.wv[word])
        except KeyError:
            missing_words.append(word)
    
    if word_vectors:
        sentence_vector = np.mean(word_vectors, axis=0)
    else:
        sentence_vector = np.zeros(vector_size)
    
    custom_vectors.append(sentence_vector)
    custom_missing_words.extend(missing_words)

# Add custom embeddings to DataFrame
sentences['custom_word2vec_embedding'] = custom_vectors

In [38]:
# Compare with pretrained model
print(f"Custom model OOV words: {len(set(custom_missing_words))}")
print(f"Sample custom OOV words: {list(set(custom_missing_words))[:10]}")

Custom model OOV words: 859
Sample custom OOV words: ['seriously?', 'longer.', 'but,', 'owe', 'cooking.', 'panini.', 'appallingly', 'position.', "he's,", 'words.']


### Additional feature

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
import pandas as pd

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Set model to evaluation mode
model.eval()

# Function to get BERT embeddings for a batch of sentences
def get_bert_embeddings(sentences, max_length=512):
   embeddings = []
   
   # Process sentences in batches to avoid memory issues
   batch_size = 8
   
   for i in range(0, len(sentences), batch_size):
       batch_sentences = sentences[i:i+batch_size]
       
       # Tokenize the batch
       encoded = tokenizer(
           batch_sentences,
           padding=True,
           truncation=True,
           max_length=max_length,
           return_tensors='pt'
       )
       
       # Get embeddings without computing gradients
       with torch.no_grad():
           outputs = model(**encoded)

           cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
           
           embeddings.extend(cls_embeddings)
   
   return np.array(embeddings)

# Convert your sentences to BERT embeddings
sentences_list = sentences['Sentence'].tolist()
bert_embeddings = get_bert_embeddings(sentences_list)

# Add BERT embeddings as a new column to your dataframe
sentences['bert_embedding'] = [emb for emb in bert_embeddings]

# Verify the results
print(f"BERT embedding shape: {bert_embeddings.shape}")
print(f"Each sentence embedding shape: {sentences['bert_embedding'].iloc[0].shape}")
print(f"DataFrame shape: {sentences.shape}")

def get_bert_embeddings_mean_pooling(sentences, max_length=512):
   embeddings = []
   batch_size = 8
   
   for i in range(0, len(sentences), batch_size):
       batch_sentences = sentences[i:i+batch_size]
       
       encoded = tokenizer(
           batch_sentences,
           padding=True,
           truncation=True,
           max_length=max_length,
           return_tensors='pt'
       )
       
       with torch.no_grad():
           outputs = model(**encoded)
           
           # Mean pooling with attention mask
           attention_mask = encoded['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
           sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, 1)
           sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
           mean_embeddings = sum_embeddings / sum_mask
           
           embeddings.extend(mean_embeddings.cpu().numpy())
   
   return np.array(embeddings)

# If you prefer mean pooling:
# bert_embeddings_mean = get_bert_embeddings_mean_pooling(sentences_list)
# df['bert_embedding_mean'] = [emb for emb in bert_embeddings_mean]

# Check memory usage and performance
print(f"Total sentences processed: {len(sentences_list)}")
print(f"Embedding dimensions: {bert_embeddings.shape[1]}")

c:\Users\filip\anaconda3\envs\Y2_BlockA\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\filip\anaconda3\envs\Y2_BlockA\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\filip\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate d

BERT embedding shape: (1038, 768)
Each sentence embedding shape: (768,)
DataFrame shape: (1038, 8)
Total sentences processed: 1038
Embedding dimensions: 768


In [49]:
sentences.head(10).to_csv(r'C:\Users\filip\Documents\GitHub\fae2-nlpr-group-group-12-1\Data\CSV\NLP_features.csv', index=False)